# 🚑🚑🚑 <span style="color: white; background-color: tomato"><b> Extração do Relatório de Acidentes </b></span></p>

      
🧩 <span style="color: chocolate"><b> 1- Configuração e controle de etapas </b></span></p>
- Importação de bibliotecas (pandas, openpyxl, pathlib etc.)
- Definição da estrutura CONFIG com caminhos:PROCESSOS.xlsx  
- Downloads  
- Arquivos Movidos  
- Base tratada (saída)
- Registro da Etapa 0 no controle de processos
- Essa etapa garante auditoria e governança da execução

📥  <span style="color: chocolate"><b> 2- Captura dos arquivos brutos do relatório de acidentes </b></span></p>
- O pipeline identifica arquivos na pasta de downloads cujo nome corresponde ao padrão “REL ACIDENTES”
- Para cada arquivo encontrado:
    - Adiciona à lista de processamento  
    - Controla múltiplos arquivos  
    - Garante integridade do fluxo antes de prosseguir

🧹 <span style="color: chocolate"><b> 3- Tratamento, limpeza e padronização da base </b></span></p>
- O script realiza leitura robusta do Excel
- Com engine openpyxl realiza mapeamento de colunas
- Transforma colunas originais em nomes padronizados utilizados nos demais projetos (ex.: registro, nome, data, setor, evento, descrição etc.)
- Conversão de tipos:
    - Datas → datetime  
    - Valores numéricos → float / int  
    - Strings → normalização
- Padronização de campos críticos:
    - Tipo de acidente  
    Classificação  
    Setor / departamento  
    Indicadores binários
- Remoção de caracteres inválidos
- Eliminação de caracteres que quebram Excel ou CSV
- O resultado é uma base limpa, normalizada e pronta para unificação

📦 <span style="color: chocolate"><b> 4- Consolidação de múltiplos arquivos </b></span></p>
- Processa todos os arquivos  
- Empilha (concat) todos os dataframes  
- Remove duplicidades  
- Reorganiza colunas em ordem fixa  
- Gera dataset consolidado
- Ao final, registra Etapa 1 no arquivo de processos

📊 <span style="color: chocolate"><b> 5- Geração da planilha final estruturada </b></span></p>
- Cria o arquivo ACIDENTES.xlsx
- Com aba única  
- Tabela estruturada Excel  
- Estilo TableStyleLight13 (linhas listradas e formato corporativo)  
- Confiabilidade para uso por áreas de RH

📁 <span style="color: chocolate"><b> 6- Movimentação dos arquivos processados </b></span></p>
- Arquivos brutos são movidos para ARQUIVOS MOVIDOS  
- Garantia de organização  
- Evita retrabalho e reprocessamento indevido  
- Mantém trilha de auditoria de insumos originais

🗃️ <span style="color: chocolate"><b> 7- Exportação para banco de dados </b></span></p>
- Geração automática do CSV tb_acidentes.csv
- A base é pronta para ingestão em:
    - SQL  
    - DW/ETL  
    - Dashboards (Power BI / Looker etc.)  
    - Modelos de People Analytics

🧾 <span style="color: chocolate"><b> 8- Registro da Etapa 2 e encerramento </b></span></p>
- Registra a conclusão (Etapa 2)  
- Calcula o tempo total de execução  
- Exibe resumo final
- Isso garante rastreabilidade, controle e governança do fluxo

# Importando as Bibliotecas

In [1]:
import pygetwindow as gw
import logging
import os
import pandas as pd
import pyautogui
import pyperclip
import shutil
import socket
import sys
import time
import tkinter as tk
import win32com.client
import win32gui, win32con
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime, date
from dotenv import load_dotenv
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import WebDriverException

tempo_0 = [id, datetime.today(), 0]

# Logging (apenas console)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [2]:
CONFIG = {
    'id_processo': 18,
    'processos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'movidos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'saida': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ACIDENTES.xlsx'),
    'env_path': Path(r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\.env'),
    'padrao': 'REL-DE-ACIDENTES',
}

# Registra Etapa do Processo

In [3]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Configurações Iniciais e Acessando o Datamace

In [4]:
load_dotenv(dotenv_path='env_path')

cloud_user = os.getenv("CLOUD_USER")
cloud_password = os.getenv("CLOUD_PASSWORD")
app_user = os.getenv("APP_USER")
app_password = os.getenv("APP_PASSWORD")
datamace = os.getenv("SITE_DATAMACE")

# Validar se todas existem
if not all([cloud_user, cloud_password, app_user, app_password]):
    raise ValueError("Uma ou mais variáveis de ambiente não foram encontradas. Verifique o arquivo .env.")

titulo_aba = "HTML5"

# Verifica se janela 'HTML5' está aberta
if gw.getWindowsWithTitle(titulo_aba):
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')
else:
    driver = webdriver.Chrome()
    time.sleep(1)
    window = win32gui.GetForegroundWindow()
    win32gui.ShowWindow(window, win32con.SW_MAXIMIZE)
    time.sleep(1)
    driver.get(datamace)
    time.sleep(8)
    # Login cloud
    driver.find_element(By.NAME, "username").send_keys(cloud_user)
    driver.find_element(By.NAME, "Password").send_keys(cloud_password)
    driver.find_element(By.NAME, "Password").send_keys(Keys.ENTER)
    time.sleep(30)
    # Login pyautogui
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')

print("🏁 Acesso finalizado")

🏁 Acesso finalizado


# Acessando o ST

In [5]:
# Fechando janelas de Novidades do Datamace
time.sleep(3)
pyautogui.press('enter')
time.sleep(3)
pyautogui.click(x=1268, y=199)
# Acessando o ST
time.sleep(10) # Tempo maior para aguardar carregar o ST
pyautogui.click(x=424, y=198) # Clicando no módulo ST
time.sleep(5)
pyautogui.click(x=1084, y=240) # Fecha a janela de Tarefas Anuais
time.sleep(2)
pyautogui.click(x=320, y=300) # Acidentes do Trabalho
time.sleep(2)
pyautogui.click(x=394, y=518) # Relatórios
time.sleep(2)
pyautogui.click(x=542, y=523) # Rel. de acidentes
time.sleep(2)
pyautogui.click(x=458, y=371) # Multi processamento
time.sleep(2)
pyautogui.click(x=846, y=373) # Inclui a empresa na classificação?
time.sleep(2)
pyautogui.click(x=636, y=409) # Situação dos trabalhadores:
time.sleep(2)
pyautogui.click(x=627, y=453) # Todos
time.sleep(2)
pyautogui.click(x=555, y=604) # Saída
time.sleep(2)
pyautogui.click(x=521, y=631) # Excel
time.sleep(2)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(2)
pyautogui.click(x=617, y=651) # Confirmar
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Extraindo relatório')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Extraindo relatório

----------------------------------------------------------------------------------------------------


# Aguardando a Conclusão da Base ACIDENTES

In [6]:
# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def verificar_download_base() -> bool:
    """
    Exibe uma caixa de diálogo para confirmar se o download da base COLAB foi realizado.
    """
    tipo_escolhido = pyautogui.confirm(
        text='Foi realizado o download da base ACIDENTES?', 
        title='Seleção de Extração', 
        buttons=['Sim']
    )

    # Se fechar a janela
    if tipo_escolhido is None:
        logging.warning("Operação cancelada pelo usuário no prompt inicial. ❌ Encerrando o script.")
        sys.exit(0)
        
    # Se for 'Sim'
    logging.info(f"Opção selecionada: {tipo_escolhido}. ✅ Continuando o processo...")
    return True

# --- Execução principal ---
if __name__ == "__main__":
    # Chama a função de validação antes de seguir com o restante do código
    verificar_download_base()
    
    logging.info("Iniciando a leitura e processamento dos dados...")

2026-06-25 09:30:01,444 - INFO - Opção selecionada: Sim. ✅ Continuando o processo...
2026-06-25 09:30:01,445 - INFO - Iniciando a leitura e processamento dos dados...


# Mapeamento de Colunas

In [7]:
MAPEAMENTO_COLUNAS = {
    'COD_EMP': 'cod_empresa',
    'REGISTRO': 'registro',
    'DC': 'dc',
    'DIRET': 'diret',
    'DEPTO': 'depto',
    'SETOR': 'setor',
    'SECAO': 'secao',
    'CCUSTO': 'cod_cc',
    'NOME ACID.': 'nome',
    'COD_CARGO': 'cod_cargo',
    'DESC_CARG': 'desc_cargo',
    'DAT_OCORR': 'data_ocorrencia',
    'HOR_OCORR': 'horario_ocorrencia',
    'TIPO ACID.': 'tipo_acidente',
    'TP_ACI_ESO': 'tp_aci_eso',
    'NUM_CAT': 'num_cat',
    'INICIO CAT': 'inicio_cat',
    'DATA CAT': 'data_cat',
    'TIPO CAT': 'tipo_cat',
    'NUM_CA_ORI': 'num_ca_ori',
    'DAT_CA_ORI': 'dat_ca_ori',
    'REC_CAT_OR': 'rec_cat_or',
    'CAUSA BASI': 'causa',
    'GRAVIDADE': 'gravidade',
    'DAT_AFAST': 'data_afastamento',
    'APOS_HORAS': 'apos_horas',
    'RETOR_CRST': 'retor_crst',
    'DIAS AFAST': 'dias_afast',
    'COD_ACID': 'cod_acidente',
    'ACIDENTE': 'acidente',
    'INCAPACIT.': 'incapacitado',
    'REG_POLICI': 'reg_policia',
    'OBITO': 'obito',
    'DAT_OBITO': 'data_obito',
    'LOCAL ACID': 'local_acidente',
    'DESC_LOCAL': 'desc_local',
    'PAIS ACID.': 'pais_acid',
    'COD_EXTER.': 'cod_exter',
    'LOC_CEP': 'cep',
    'LOC_ENDERE': 'endereco',
    'LOC_NUM.': 'num',
    'LOC_COMPL': 'compl',
    'LOC_BAIRRO': 'bairro',
    'LOC_UF': 'uf',
    'LOC_CIDADE': 'cidade',
    'LOC_TP_INS': 'loc_tp_ins',
    'LOC_INSC.': 'insc',
    'DESC_ACID': 'descricao_acidente',
    'PAR_COR_AT': 'parte_corpo',
    'PAR_ATIN1': 'par_atin1',
    'LATERAL_1': 'lateral1',
    'PAR_ATIN2': 'par_atin2',
    'LATERAL_2': 'lateral2',
    'PAR_ATIN3': 'par_atin3',
    'LATERAL_3': 'lateral3',
    'PAR_ATIN4': 'par_atin4',
    'LATERAL_4': 'lateral4',
    'PAR_ATIN5': 'par_atin5',
    'LATERAL_5': 'lateral5',
    'PAR_ATIN6': 'par_atin6',
    'LATERAL_6': 'lateral6',
    'PAR_ATIN7': 'par_atin7',
    'LATERAL_7': 'lateral7',
    'PAR_ATIN8': 'par_atin8',
    'LATERAL_8': 'lateral8',
    'PAR_ATIN9': 'par_atin9',
    'LATERAL_9': 'lateral9',
    'PAR_ATIN10': 'par_atin10',
    'LATERAL_10': 'lateral10',
    'SIT_GERAD.': 'sit_gerad',
    'AG_CAUSA_1': 'ag_causa1',
    'AG_CAUSA_2': 'ag_causa2',
    'AG_CAUSA_3': 'ag_causa3',
    'AG_CAUSA_4': 'ag_causa4',
}
COLUNAS_DATA = ['DAT_OCORR', 'DATA CAT', 'DAT_AFAST', 'DAT_OBITO']

# Processando Acidentes

In [8]:
@contextmanager

def gerenciar_workbook(caminho: Path, sheet: str):
    """Context manager para operações seguras com workbook."""
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)

def registrar_etapa(caminho: Path, id_proc: int, etapa: int):
    """Registra etapa do processo."""
    with gerenciar_workbook(caminho, 'REGISTROS') as ws:
        ws.append([id_proc, datetime.today(), etapa])
    logger.debug(f"✅ Etapa {etapa} registrada")

def converter_datas(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Converte colunas para datetime."""
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')
    return df

def mapear_colunas_acidentes(df: pd.DataFrame) -> pd.DataFrame:
    """Mapeia colunas do arquivo de acidentes."""
    colunas_existentes = {k: v for k, v in MAPEAMENTO_COLUNAS.items() if k in df.columns}
    return df.rename(columns=colunas_existentes)[list(colunas_existentes.values())]

def processar_arquivo(caminho: Path) -> pd.DataFrame:
    """Processa um arquivo de acidentes."""
    try:
        logger.debug(f"Carregando: {caminho.name}")
        
        # Carregar
        df = pd.read_excel(caminho, engine='openpyxl')
        
        # Limpar
        df = df.dropna(subset=['NOME ACID.'])
        
        # Converter tipos
        df['REGISTRO'] = pd.to_numeric(df['REGISTRO'], errors='coerce')
        df = converter_datas(df, COLUNAS_DATA)
        
        # Mapear colunas
        df = mapear_colunas_acidentes(df)
        
        logger.debug(f"✅ {len(df)} registros processados")
        return df
        
    except Exception as e:
        logger.error(f"❌ Erro ao processar {caminho.name}: {e}")
        return None
    
def criar_excel_com_tabela(df: pd.DataFrame, caminho: Path):
    """Cria Excel com tabela formatada."""
    logger.info(f"📝 Criando Excel final ({len(df)} registros)...")
    
    # Salvar com Pandas
    df.to_excel(caminho, sheet_name='ACIDENTES', index=False, engine='openpyxl')
    
    # Aplicar formatação
    wb = load_workbook(caminho)
    ws = wb.active
    
    tabela = Table(displayName="ACIDENTES", ref=ws.dimensions)
    estilo = TableStyleInfo(
        name="TableStyleLight13",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela.tableStyleInfo = estilo
    ws.add_table(tabela)
    
    wb.save(caminho)
    logger.info(f"✅ Excel criado: {caminho}")
    
# Main
if __name__ == "__main__":
    tempo_inicio = datetime.now()
    
    logger.info("=" * 80)
    logger.info("🔄 PROCESSAMENTO DE ACIDENTES")
    logger.info("=" * 80)

2026-06-25 09:30:01,494 - INFO - ================================================================================
2026-06-25 09:30:01,497 - INFO - 🔄 PROCESSAMENTO DE ACIDENTES
2026-06-25 09:30:01,500 - INFO - ================================================================================


# Processamento e Consolidação do Arquivo

In [9]:
try:
    # Etapa 1: Registrar início
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 0)
        
    # Etapa 2: Buscar arquivos
    logger.info("\n📂 Buscando arquivos...")
    arquivos = [f for f in CONFIG['origem'].iterdir() if f.name.startswith(CONFIG['padrao'])]
        
    if not arquivos:
        logger.warning("❌ Nenhum arquivo encontrado")
        exit()
        
    logger.info(f"✅ {len(arquivos)} arquivo(s) encontrado(s)\n")
        
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 1)
        
    # Etapa 3: Processar arquivos
    logger.info("🔄 Processando arquivos...\n")
        
    todas_bases = []
        
    for idx, arquivo in enumerate(arquivos, 1):
        logger.info(f"[{idx}/{len(arquivos)}] {arquivo.name}")
            
        df = processar_arquivo(arquivo)
            
        if df is not None and len(df) > 0:
            todas_bases.append(df)
                
            try:
                destino = CONFIG['movidos'] / arquivo.name
                shutil.move(str(arquivo), str(destino))
                logger.info(f"✅ Movido para arquivos processados\n")
            except Exception as e:
                logger.error(f"⚠️ Erro ao mover: {e}\n")
        else:
            logger.warning(f"❌ Sem dados válidos\n")
        
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 2)
        
    # Etapa 4: Consolidar e salvar
    if todas_bases:
        logger.info("💾 Consolidando dados...")
        base_final = pd.concat(todas_bases, ignore_index=True)
        base_final = base_final.drop_duplicates()
        logger.info(f"✅ {len(base_final)} registros consolidados\n")
            
        criar_excel_com_tabela(base_final, CONFIG['saida'])
    else:
        logger.error("❌ Nenhum arquivo foi processado!")
        exit()
        
    # Finalizar
    tempo_duracao = datetime.now() - tempo_inicio
        
    logger.info("\n" + "=" * 80)
    logger.info("✅ PROCESSO FINALIZADO COM SUCESSO!")
    logger.info(f"   Tempo de execução: {tempo_duracao}")
    logger.info("=" * 80)
        
except Exception as e:
    logger.error(f"\n❌ ERRO CRÍTICO: {e}")
    import traceback
    logger.error(traceback.format_exc())

2026-06-25 09:30:02,027 - INFO - 
📂 Buscando arquivos...
2026-06-25 09:30:02,028 - INFO - ✅ 1 arquivo(s) encontrado(s)

2026-06-25 09:30:02,354 - INFO - 🔄 Processando arquivos...

2026-06-25 09:30:02,354 - INFO - [1/1] REL-DE-ACIDENTES-09294296.XLSX
c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
2026-06-25 09:30:02,647 - INFO - ✅ Movido para arquivos processados

2026-06-25 09:30:03,006 - INFO - 💾 Consolidando dados...
2026-06-25 09:30:03,015 - INFO - ✅ 150 registros consolidados

2026-06-25 09:30:03,016 - INFO - 📝 Criando Excel final (150 registros)...
2026-06-25 09:30:03,587 - INFO - ✅ Excel criado: X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ACIDENTES.xlsx
2026-06-25 09:30:03,587 - INFO - 
2026-06-25 09:30:03,588 - INFO - ✅ PROCESSO FINALIZADO COM SUCES

# Atualizando o Arquivo Excel Controle HC e Atestados

In [10]:
# Caminho do arquivo
path_excel = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
os.startfile(path_excel) # Abre o arquivo
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('ACIDENTES!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

#Atualizando o arquivo
pyautogui.hotkey('alt', 'F5')
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Criando Base em CSV para o Banco de Dados

In [11]:
# Ler o XLSX
df = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ACIDENTES.xlsx')

# Salvar como CSV
df.to_csv(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\tb_acidentes.csv', index=False, encoding='utf-8')

print("✅ Arquivo convertido para CSV com sucesso!")

✅ Arquivo convertido para CSV com sucesso!


# Resumo de Finalização do Processo

In [12]:
tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:02:30.104537

----------------------------------------------------------------------------------------------------
